# Evaluating segmentation models

In `training.ipynb` we trained a model that finds nuclei in the phalloidin channel. Is it
any good? "It looks fine" is not an answer, and neither is a single number without knowing
what it measures.

In this notebook we put three generic, pretrained models next to the model we trained,
on the same test images, and score them all in the same way.

Run this notebook with the **day 2 deep learning** kernel (`2026_deep_learning`): it
needs StarDist and Cellpose. The BiaPy model is not run here, we only read the
predictions it wrote to disk during training.

## Learning goals

- Explain the difference between a generic pretrained model and a model trained for one task
- Explain intersection over union (IoU) and how it decides whether two objects match
- Calculate and interpret precision, recall and F1
- Choose the metric that fits the biological question

## Generic and specific models

On day 2 we used **generic** models. They were trained on thousands of images from many
labs, so they work on many kinds of images, as long as your images look like what they
were trained on.

| | Generic pretrained model | Model trained for your task |
| --- | --- | --- |
| Examples | StarDist `2D_versatile_fluo`, Cellpose `cyto3` and `nuclei` | the BiaPy model from `training.ipynb` |
| Training data | large and varied, someone else's | small, yours |
| What it finds | what it was trained to find: nuclei, cells | whatever your labels show |
| Your effort | choose the model, set size and normalization | make labels, train, check |
| Fails when | your images or objects differ from its training data | your labels are wrong, or new images differ from your training set |

Our task is one no generic model was trained for: find the **nuclei** in an image of
**actin**. A nuclei model expects a bright DAPI-like signal; a cell model finds the cell
outlines that actin shows. Neither answers our question exactly. Let's measure how far
off they are.

## Libraries

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from skimage.measure import regionprops

from stardist.models import StarDist2D
from stardist.matching import matching, matching_dataset
from stardist import random_label_cmap
from csbdeep.utils import normalize
from cellpose import models

lbl_cmap = random_label_cmap()

ModuleNotFoundError: No module named 'matplotlib'

## The test set

We use the test images that `training.ipynb` held back. The trained model has never
seen them, and neither have the pretrained models.

Note what the labels are: StarDist nuclei from the **DAPI** channel. They are not drawn
by hand, so they are a *silver standard*. Every score below measures agreement with
these labels, not with the truth.

In [ ]:
TEST = Path('dataset/test')

names = sorted(p.name for p in (TEST / 'images').glob('*.tif'))
images = [tifffile.imread(TEST / 'images' / n) for n in names]
ground_truth = [tifffile.imread(TEST / 'labels' / n) for n in names]

print(len(names), 'test images, shape', images[0].shape)

In [ ]:
# The nuclei model in Cellpose needs the size of the nuclei, so we measure it
diameter = np.median([p.equivalent_diameter_area for gt in ground_truth for p in regionprops(gt)])
print(f'median nucleus diameter: {diameter:.1f} pixels')

## Running three generic models

- **StarDist** `2D_versatile_fluo` — nuclei in fluorescence
- **Cellpose** `nuclei` — nuclei, told the diameter we just measured
- **Cellpose** `cyto3` — cells; it estimates the size itself

This takes a minute on a CPU.

In [ ]:
stardist_model = StarDist2D.from_pretrained('2D_versatile_fluo')
cellpose_nuclei = models.CellposeModel(model_type='nuclei', gpu=False)
cellpose_cyto3 = models.CellposeModel(model_type='cyto3', gpu=False)

predictions = {
    'StarDist': [stardist_model.predict_instances(normalize(im, 1, 99.8))[0] for im in images],
    'Cellpose nuclei': [cellpose_nuclei.eval(im, channels=[0, 0], diameter=diameter)[0] for im in images],
    'Cellpose cyto3': [cellpose_cyto3.eval(im, channels=[0, 0], diameter=None)[0] for im in images],
}

In [ ]:
i = 0  # change this to look at another test image

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(images[i], cmap='gray')
axes[0].imshow(ground_truth[i], cmap=lbl_cmap, alpha=0.4)
axes[0].set_title(f'labels: {ground_truth[i].max()} nuclei')
for ax, (model, labels) in zip(axes[1:], predictions.items()):
    ax.imshow(images[i], cmap='gray')
    ax.imshow(labels[i], cmap=lbl_cmap, alpha=0.4)
    ax.set_title(f'{model}: {labels[i].max()} objects')
for ax in axes:
    ax.axis('off')

Before any numbers: which model would you trust, and for what?

## Metric 1: counting objects

The simplest score is the number of objects. If your question is "how many cells are in
this well", it may be all you need.

In [ ]:
counts = pd.DataFrame({'labels': [gt.max() for gt in ground_truth]}, index=names)
for model, labels in predictions.items():
    counts[model] = [lbl.max() for lbl in labels]
counts

One of the models gets the count almost right. Look back at the images: is it finding
nuclei? A count says nothing about *where* the objects are or what their shape is.

## Metric 2: intersection over union

To know whether a predicted object is the same as a labelled one, we compare their
pixels. **Intersection over union (IoU)** is the number of pixels they share, divided by
the number of pixels they cover together:

- 1.0 — identical
- 0.5 — reasonable overlap
- 0.0 — no overlap at all

Two circles with a shifted centre show what that looks like:

In [ ]:
yy, xx = np.mgrid[:100, :100]
labelled = (yy - 50) ** 2 + (xx - 40) ** 2 < 25 ** 2
predicted = (yy - 50) ** 2 + (xx - 60) ** 2 < 25 ** 2

intersection = (labelled & predicted).sum()
union = (labelled | predicted).sum()

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, img, title in zip(axes,
                          [labelled + 2 * predicted, labelled & predicted, labelled | predicted],
                          ['labelled (1), predicted (2)', f'intersection: {intersection} px', f'union: {union} px']):
    ax.imshow(img, cmap='viridis')
    ax.set_title(title)
    ax.axis('off')
fig.suptitle(f'IoU = {intersection} / {union} = {intersection / union:.2f}')

### Exercise

Move the predicted circle closer (`xx - 50`) or make it larger (`30 ** 2`). How far off
can a prediction be and still have an IoU above 0.5?

## Metric 3: precision, recall and F1

With IoU we can match objects: a labelled and a predicted object are the **same** object
when their IoU is above a threshold, usually 0.5. Every object then falls in one of three
groups:

- **true positive (TP)** — a predicted object that matches a labelled one
- **false positive (FP)** — a predicted object that matches nothing: something extra
- **false negative (FN)** — a labelled object that nothing matches: something missed

From these three numbers:

- **precision** = TP / (TP + FP) — of what the model found, how much is right?
- **recall** = TP / (TP + FN) — of what is there, how much did the model find?
- **F1** = 2 · precision · recall / (precision + recall) — both in one number

StarDist has a function for this, `matching`, and it works on any label image, also
from Cellpose or BiaPy.

In [ ]:
score = matching(ground_truth[0], predictions['StarDist'][0], thresh=0.5)
print(f'TP {score.tp}  FP {score.fp}  FN {score.fn}')
print(f'precision {score.precision:.2f}  recall {score.recall:.2f}  F1 {score.f1:.2f}')
print(f'mean IoU of the matched objects {score.mean_matched_score:.2f}')

Now all models over the whole test set. `matching_dataset` adds up TP, FP and FN over all
images before calculating the scores.

In [ ]:
def score_table(predictions, thresh=0.5):
    rows = []
    for model, labels in predictions.items():
        s = matching_dataset(ground_truth, labels, thresh=thresh, show_progress=False)
        rows.append({'model': model, 'TP': s.tp, 'FP': s.fp, 'FN': s.fn,
                     'precision': s.precision, 'recall': s.recall, 'F1': s.f1,
                     'mean matched IoU': s.mean_matched_score})
    return pd.DataFrame(rows).set_index('model').round(2)


score_table(predictions)

- Which model has a high precision but a low recall? What does that mean in the image?
- Which model got the count right but has the lowest F1? Why?

## The threshold is a choice

An IoU threshold of 0.5 is a convention, not a law. Let's see how the F1 changes when we
are more lenient or more strict.

In [ ]:
thresholds = np.arange(0.1, 1.0, 0.1)

fig, ax = plt.subplots(figsize=(6, 4))
for model, labels in predictions.items():
    f1 = [matching_dataset(ground_truth, labels, thresh=t, show_progress=False).f1 for t in thresholds]
    ax.plot(thresholds, f1, marker='o', label=model)
ax.set_xlabel('IoU threshold')
ax.set_ylabel('F1')
ax.legend()

A model whose F1 is high at a low threshold and collapses at a higher one finds the right
objects in roughly the right place, but with the wrong outline. Which model is that, and
what is it outlining?

## Adding the model we trained

During training BiaPy ran the trained model on the same test images and wrote the
predicted instances to disk. Set `job_name` and `run_id` to the run you want to score.

In [ ]:
output_path = Path('biapy_output')
job_name = 'nuclei_from_phalloidin'
run_id = 1

biapy_results = output_path / job_name / 'results' / f'{job_name}_{run_id}' / 'per_image_instances'

if biapy_results.is_dir():
    predictions['BiaPy (trained)'] = [tifffile.imread(biapy_results / n) for n in names]
    print('added the BiaPy predictions from', biapy_results)
else:
    print('No predictions found in', biapy_results)
    print('Let the training in training.ipynb finish, with TEST.ENABLE set to True.')

In [ ]:
score_table(predictions)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for model, labels in predictions.items():
    f1 = [matching_dataset(ground_truth, labels, thresh=t, show_progress=False).f1 for t in thresholds]
    ax.plot(thresholds, f1, marker='o', label=model)
ax.set_xlabel('IoU threshold')
ax.set_ylabel('F1')
ax.legend()

## Discussion: which metric fits the question?

There is no best metric, only a metric that fits what you want to measure afterwards.

| Biological question | What has to be right | Metric |
| --- | --- | --- |
| How many cells per well? | the number of objects | count, or F1 at a low IoU threshold |
| Does treatment change nuclear size or shape? | the outline of each object | mean matched IoU, F1 at a high threshold |
| Intensity of a marker per nucleus? | which pixels belong to which nucleus | F1 at 0.5 plus mean matched IoU |
| Are rare cells missed? | every object found | recall |

- For each question, which model would you pick from the table above?
- Did training a specific model pay off, compared with the best generic one?
- The labels come from StarDist on DAPI. Could a model ever score better than those
  labels? What would you need to find out?

## Exercises

1. **Look at the errors.** Pick the model with the lowest recall and show, for one image,
   the labels it missed. Hint: `matching(..., report_matches=True)` returns which labels
   were matched.
2. **Help the generic model.** Cellpose `nuclei` got a diameter from us. Try twice and
   half that value. Does recall change? Is it a size problem or a channel problem?
3. **Compare training runs.** Score the runs you made in `training.ipynb` with other
   settings (for example `no_augmentation`) by changing `job_name`. Which setting made the
   largest difference?